In [3]:
#必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

#環境変数の取得
load_dotenv("../.env")

#OpenAi APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

#tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

#モデル名
MODEL_NAME ="gpt-4o-mini"

In [2]:
pip install tavily-python

  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
#検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response =client.search(question)
    return json.dumps({"result":response["results"]})


In [5]:
#テストコード
ret = get_search_result("東京駅のイベントを教えて")
json.loads(ret)


{'result': [{'url': 'https://ekitan.com/event/station-2590',
   'title': '東京駅周辺のイベント',
   'content': '東京駅のイベント一覧 ; Vision Network マルシェ. 女性首長のまちからご当地品をお届け！ · 2026年10月15日(木)～10月19日(月) ; The CANDLE TABLE. ミュージカル俳優が贈る',
   'score': 0.75852895,
   'raw_content': None,
   'id': '303609-00'},
  {'url': 'https://www.enjoytokyo.jp/event/list/area1306',
   'title': '東京駅周辺・丸の内でおすすめのイベント',
   'content': '国内外のチョコレートブランドが集まり、世界各地のカカオやチョコレートを楽しめるイベント「Tokyo Chocolate Salon 2026（東京チョコレートサロン）」が、丸の内で開催！\n\n  + 東京ビル TOKIA\n  + 東京\n\n  グルメ   スイーツ\n 東京都\u3000港区\n\n  DESIGNART TOKYO 2026\n\n  2026/10/30(金) ～ 11/08(日)\n\n  DESIGNART TOKYO 2026\n\n  日本最大級のデザイン＆アートフェスティバル！第10回開催となる今年のテーマは「Hibiki」\n\n  + 銀座駅（東京メトロ） / 外苑前駅（東京メトロ） / 東京駅（東京メトロ） / 表参道駅（東京メトロ） / 六本木駅（東京メトロ） / 原宿駅（JR東日本） / 渋谷駅（JR東日本）\n  + 銀座   / 外苑前   / 東京   / 表参道   / 六本木   / 原宿   / 渋谷\n\n  体験・遊び\n 東京都\u3000千代田区\n\n  なめこ栽培キット15周年記念 POP UP STORE んふんふパーティ♪ [...] 平均予算\u30005,000円\n\nTHE ROAST KOBE Meat House\n\n東京駅／東京都  有楽町1分の天空テラス♪/飲み放題付5000円～ \n\n有楽町1分

In [17]:
#ツール定義
def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type":"function",
            "function":{
                "name":"get_search_result",
                "description":"最近一カ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters":{
                    "type":"object",
                    "properties":{
                        "question":{"type":"string","description":"質問文"},
                    },
                    "required":["question"],
                },
            },
        })
    ]

In [18]:
#言語モデルへの質問を行う関数
def ask_question(question,tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role":"user","content":question}],
        tools=tools,
        tool_choice="auto"
    )
    return response

In [19]:
# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

In [20]:
# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()


In [21]:
tools = define_tools()

# 言語モデルが直接回答できる質問
question = "東京都と沖縄県はどちらが広いですか？"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
東京都と沖縄県の面積を比較すると、沖縄県の方が広いです。

- 東京都の面積は約2,194平方キロメートルです。
- 沖縄県の面積は約2,271平方キロメートルです。

したがって、沖縄県は東京都よりも約77平方キロメートル広いことになります。


In [22]:
tools = define_tools()

#ツール呼び出しが必要な質問
question = "東京駅のイベントについて、最近1カ月以内の検索結果を教えてください"
response_message = process_response(question,tools)
print(response_message)

------define_tools(ツール定義)------
最近の東京駅周辺でのイベント情報は以下の通りです。

1. **Tokyo Chocolate Salon 2026**
   - **期間**: 2026年10月4日（木）
   - **場所**: 東京駅周辺
   - **内容**: 国内外のチョコレートブランドが集まり、試食や販売が行われるイベントです。
   - [詳細はこちら](https://www.enjoytokyo.jp/event/list/area1306)

2. **TBSスタジオ出張所「TEAM JAPAN × ポムポムプリン コラボレーション POP-UP STORE」**
   - **期間**: 2026年9月25日（金）〜2026年10月8日（木）
   - **場所**: 東京駅 一番街 地下1階
   - [詳細はこちら](https://www.tokyoeki-1bangai.co.jp/event)

3. **PEANUTS I LIKE...**
   - **期間**: 2026年9月18日（金）〜2026年10月1日（木）
   - **場所**: 東京駅 一番街の地下1階
   - 内容: PEANUTS関連のグッズが販売されます。
   - [詳細はこちら](https://www.tokyoeki-1bangai.co.jp/event)

4. **「よーじや」POP-UP SHOP**
   - **期間**: 2026年9月25日（金）〜2026年10月1日（木）
   - **場所**: 東京駅 一番街 地下1階
   - 内容: 京都を代表する化粧品店の特設ショップです。
   - [詳細はこちら](https://www.tokyocity-i.jp/event)

5. **「八重洲地下街」イベント**
   - **期間**: 2026年9月30日（水）〜2026年10月12日（月）
   - **場所**: 八重洲地下街
   - 内容: 地元のアートや文化を紹介するイベント（テーマは「秋の特別展示」）。
   - [詳細はこちら](https://www.gotokyo.org/jp/calendar/index.html)

これらのイベントは、東京駅の近くで楽しむことが

In [ ]:
# チャットボットへの組み込み
tools = define_tools()

messages=[]

while(True):
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip()=="":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:こんにちは'

こんにちは！今日はどんなことをお手伝いできますか？


'質問:東北6県は'

東北6県とは、以下の県を指します。

1. 青森県
2. 岩手県
3. 宮城県
4. 秋田県
5. 山形県
6. 福島県

これらの県は日本の東北地方に位置し、豊かな自然と歴史的な文化が特徴です。何か特定の情報や質問がありますか？

---ご利用ありがとうございました！---
